# HANDLING MISSING DATA


## Bước 1: Raw Data

In [1]:
import pandas as pd
import numpy as np

data = {
    'ID': ['KH001', 'KH002', 'KH003', 'KH004', 'KH005', 'KH006'],
    'Ho_Ten': ['Nguyen Van A', 'Tran Thi B', 'Le Thi C', 'Pham Minh D', 'Vu Hoang E', 'Hoang Ngoc F'],
    'Gioi_Tinh': ['Nam', 'Nữ', 'Nữ', 'Nam', 'Nam', 'Nữ'],
    'Can_Nang': [72.0, np.nan, np.nan, 65.0, 80.0, np.nan],  
    'Thu_Nhap': [15.0, 18.0, 12.0, np.nan, 8.0, np.nan]    
}

df = pd.DataFrame(data)
df

,ID,Ho_Ten,Gioi_Tinh,Can_Nang,Thu_Nhap
0,KH001,Nguyen Van A,Nam,72.0,15.0
1,KH002,Tran Thi B,Nữ,NaN,18.0
2,KH003,Le Thi C,Nữ,NaN,12.0
3,KH004,Pham Minh D,Nam,65.0,NaN
4,KH005,Vu Hoang E,Nam,80.0,8.0
5,KH006,Hoang Ngoc F,Nữ,NaN,NaN


## Bước 2: Lưu vết trước khi xử lý (Missing Indicator Flags)

In [2]:
df['Can_Nang_Is_Missing'] = df['Can_Nang'].isna()
df['Thu_Nhap_Is_Missing'] = df['Thu_Nhap'].isna()

df[['ID', 'Ho_Ten', 'Can_Nang', 'Can_Nang_Is_Missing', 'Thu_Nhap', 'Thu_Nhap_Is_Missing']]

,ID,Ho_Ten,Can_Nang,Can_Nang_Is_Missing,Thu_Nhap,Thu_Nhap_Is_Missing
0,KH001,Nguyen Van A,72.0,False,15.0,False
1,KH002,Tran Thi B,NaN,True,18.0,False
2,KH003,Le Thi C,NaN,True,12.0,False
3,KH004,Pham Minh D,65.0,False,NaN,True
4,KH005,Vu Hoang E,80.0,False,8.0,False
5,KH006,Hoang Ngoc F,NaN,True,NaN,True


## Bước 3: Imputation

### Trường hợp 1: Xử lý cơ chế MCAR (Thiếu hoàn toàn ngẫu nhiên - Khách KH002)
* **Độ đo sử dụng:** Xu hướng trung tâm (**Mean** cho dữ liệu phân phối chuẩn, **Median** cho dữ liệu lệch/có outlier).
* **Giải pháp:** Vì cân nặng phân phối khá đồng đều, ta tính số trung bình tổng (`Mean`) để điền khuyết.

In [6]:
df_mcar = df.copy()

# Tính cân nặng trung bình của những người đã điền thông tin
mean_weight = df_mcar['Can_Nang'].mean()
print(f"[MCAR] Giá trị độ đo Mean dùng để điền khuyết: {mean_weight:.1f} kg\n")

# # Fill giá trị trung bình vào ô trống ngẫu nhiên
df_mcar['Can_Nang'] = df_mcar['Can_Nang'].fillna(mean_weight)
df_mcar[['ID', 'Ho_Ten', 'Can_Nang']]

[MCAR] Giá trị độ đo Mean dùng để điền khuyết: 72.3 kg



,ID,Ho_Ten,Can_Nang
0,KH001,Nguyen Van A,72.000000
1,KH002,Tran Thi B,72.333333
2,KH003,Le Thi C,72.333333
3,KH004,Pham Minh D,65.000000
4,KH005,Vu Hoang E,80.000000
5,KH006,Hoang Ngoc F,72.333333


### Trường hợp 2: Xử lý cơ chế MAR (Thiếu ngẫu nhiên có điều kiện - Khách KH003 & KH006)
* **Độ đo sử dụng:** Kỳ vọng có điều kiện (**Conditional Mean/Median**), cụ thể là `Groupby` theo biến bối cảnh có liên quan.
* **Giải pháp:** Nhóm theo cột `Gioi_Tinh` rồi lấy trung bình của riêng nhóm Nữ điền cho khách hàng Nữ.

In [4]:
df_mar = df.copy()

# Sử dụng hàm biến đổi dựa trên phân nhóm (Group-by Imputation)
# df_mar['Can_Nang'] = df_mar.groupby('Gioi_Tinh')['Can_Nang'].transform(lambda x: x.fillna(x.mean()))

df_mar[['ID', 'Ho_Ten', 'Gioi_Tinh', 'Can_Nang']]

,ID,Ho_Ten,Gioi_Tinh,Can_Nang
0,KH001,Nguyen Van A,Nam,72.0
1,KH002,Tran Thi B,Nữ,NaN
2,KH003,Le Thi C,Nữ,NaN
3,KH004,Pham Minh D,Nam,65.0
4,KH005,Vu Hoang E,Nam,80.0
5,KH006,Hoang Ngoc F,Nữ,NaN


### Trường hợp 3: Xử lý cơ chế NMAR (Thiếu không ngẫu nhiên - Khách KH004 & KH006)
* **Độ đo/Kỹ thuật sử dụng:** Mã hóa danh mục đặc trưng (**Constant Encoding**), giữ nguyên vết khuyết để làm tệp phân tích riêng.

In [5]:
df_nmar = df.copy()

df_nmar['Thu_Nhap'] = df_nmar['Thu_Nhap'].astype(str)
df_nmar['Thu_Nhap'] = df_nmar['Thu_Nhap'].replace('nan', 'Chưa rõ')

df_nmar[['ID', 'Ho_Ten', 'Thu_Nhap']]

,ID,Ho_Ten,Thu_Nhap
0,KH001,Nguyen Van A,15.0
1,KH002,Tran Thi B,18.0
2,KH003,Le Thi C,12.0
3,KH004,Pham Minh D,NaN
4,KH005,Vu Hoang E,8.0
5,KH006,Hoang Ngoc F,NaN
